# Import Onda A-mode measurements

Run all cells to load the `A-mode*` folders in `data/Measurements - Onda hydrophone data - 2026`.
Requires pandas and an IPython/Jupyter Python kernel. Works when launched from the project root or `src`.

- `average_df`: saved averaged readings, including the detrended amplitude and A-mode envelope.
- `pulses_df`: individual pulse readings, identified by `measurement` and `pulse_index`.
- `metadata_df`: acquisition metadata from each file's comment header.

Each waveform row represents one time sample. Original CSV column names and units are preserved (seconds and volts). The saved average is detrended, whereas pulse amplitudes are the recorded waveforms; they need not match even for a single pulse. Voltage columns are retained. The calibration cells add pressure columns in Pa using the sensitivity at a user-specified frequency for each recording.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the project when the kernel starts in the repository or src directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "Measurements - Onda hydrophone data - 2026").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook from the project root or src directory.")

measurement_root = project_root / "data" / "Measurements - Onda hydrophone data - 2026"
measurement_dirs = sorted(path for path in measurement_root.glob("A-mode*") if path.is_dir())
if not measurement_dirs:
    raise FileNotFoundError(f"No A-mode folders found in {measurement_root}")

[p.name for p in measurement_dirs]

['A-mode_2MHz_focus_10Vpp_20cycles', 'A-mode_7p37MHz_focus_10Vpp_20cycles']

In [2]:
def read_a_mode_csv(path):
    """Read waveform samples and retain the acquisition header separately."""
    metadata = {}
    with path.open(encoding="utf-8-sig") as stream:
        for line in stream:
            if not line.startswith("#"):
                break
            key, separator, value = line[1:].strip().partition(":")
            if separator:
                metadata[key.strip()] = value.strip()

    waveform = pd.read_csv(path, comment="#", encoding="utf-8-sig")
    required = (
        {"Time_s", "Average_Detrended_Amplitude_V", "A_Mode_Envelope_V"}
        if path.name == "a_scan_average.csv" else {"Time_s", "Amplitude_V"}
    )
    missing = required - set(waveform.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    for column in required:
        waveform[column] = pd.to_numeric(waveform[column], errors="raise")
    waveform.insert(0, "measurement", path.parent.name)
    waveform.insert(1, "source_file", str(path.relative_to(measurement_root)))
    metadata.update(measurement=path.parent.name,
                    source_file=str(path.relative_to(measurement_root)))
    return waveform, metadata


average_frames, pulse_frames, metadata_rows = [], [], []
for folder in measurement_dirs:
    average_path = folder / "a_scan_average.csv"
    pulse_paths = sorted(folder.glob("a_scan_pulse_*.csv"),
                         key=lambda path: int(path.stem.rsplit("_", 1)[1]))
    if not average_path.is_file() or not pulse_paths:
        raise FileNotFoundError(f"Expected average and per-pulse CSV files in {folder}")

    frame, metadata = read_a_mode_csv(average_path)
    average_frames.append(frame)
    metadata_rows.append({**metadata, "reading_type": "average"})
    for path in pulse_paths:
        frame, metadata = read_a_mode_csv(path)
        pulse_index = int(path.stem.rsplit("_", 1)[1])
        frame.insert(2, "pulse_index", pulse_index)
        pulse_frames.append(frame)
        metadata_rows.append({**metadata, "pulse_index": pulse_index,
                              "reading_type": "pulse"})

average_df = pd.concat(average_frames, ignore_index=True)
pulses_df = pd.concat(pulse_frames, ignore_index=True)
metadata_df = pd.DataFrame(metadata_rows)
metadata_df["pulse_index"] = pd.to_numeric(metadata_df["pulse_index"]).astype("Int64")

In [3]:
# Sample counts and previews of the imported DataFrames.
display(average_df.groupby("measurement", sort=False).size().rename("average_samples").to_frame())
display(pulses_df.groupby(["measurement", "pulse_index"], sort=False).size().rename("pulse_samples").to_frame())
display(average_df.head())
display(pulses_df.head())
display(metadata_df)

,average_samples
measurement,
A-mode_2MHz_focus_10Vpp_20cycles,10002
A-mode_7p37MHz_focus_10Vpp_20cycles,2002


,,pulse_samples
measurement,pulse_index,
A-mode_2MHz_focus_10Vpp_20cycles,1,10002
A-mode_7p37MHz_focus_10Vpp_20cycles,1,2002


,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-2.020624e-07,-0.002025,0.000051
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.920624e-07,0.005975,0.000047
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.820624e-07,0.001975,0.000053
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.720624e-07,-0.006024,0.000049
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.620624e-07,0.001976,0.000053


,measurement,source_file,pulse_index,Time_s,Amplitude_V
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-2.020624e-07,0.000
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.920624e-07,0.008
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.820624e-07,0.004
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.720624e-07,-0.004
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.620624e-07,0.004


,run_timestamp,run_id,mode,x_mm,y_mm,z_mm,frequency_hz,amplitude_v,cycles_per_pulse,pulse_count,...,live_preview,signal_source,effective_filter_sampling_rate_hz,effective_highpass_cutoff_hz,measurement,source_file,reading_type,pulse_index,pulse_total,scope_sampling_rate_hz
0,2026-09-25 20:02:38,2,INC,0.5,0.0,0.0,2000000.0,10.0,20.0,1,...,False,hardware,100000000.60774711,500000.0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,average,<NA>,NaN,NaN
1,2026-09-25 20:02:38,2,INC,0.5,0.0,0.0,2000000.0,10.0,20.0,1,...,False,hardware,NaN,NaN,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,pulse,1,1,100000000.60774711
2,2026-09-25 20:17:25,1,INC,0.0,0.0,0.0,7370000.0,10.0,20.0,1,...,False,hardware,100000000.60774711,500000.0,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,average,<NA>,NaN,NaN
3,2026-09-25 20:17:25,1,INC,0.0,0.0,0.0,7370000.0,10.0,20.0,1,...,False,hardware,NaN,NaN,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,pulse,1,1,100000000.60774711


## Hydrophone calibration: voltage to pressure

Edit `measurement_frequencies_mhz` below to specify the measurement frequency in **MHz** for each recording. The calibration file is read from that recording's folder (with or without a `.txt` extension).

Use the table's **Sens(V/Pa)** column directly. For a frequency between table entries, linearly interpolate sensitivity between the closest lower and higher frequencies. Frequencies outside the calibrated range raise an error.

Pressure is `amplitude_V / sensitivity_V_per_Pa`. This applies one sensitivity at the specified frequency to the whole waveform, including the envelope; it is a single-frequency conversion, without broadband frequency-response correction. The calibration file includes the AG-2010 preamplifier, so no extra amplifier gain factor is applied.

In [4]:
# USER SETTINGS: measurement frequencies in MHz (edit these values).
measurement_frequencies_mhz = {
    "A-mode_2MHz_focus_10Vpp_20cycles": 2.0,
    "A-mode_7p37MHz_focus_10Vpp_20cycles": 7.37,
}
calibration_basename = "HGL0400-2875_AG-2010-1384-20_xx_OndaCombineCal_20230621-Reza-Oct-2026"

In [5]:
import io
import numpy as np


def read_calibration(path):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header_end = next((i for i, line in enumerate(lines)
                       if line.strip().startswith("HEADER_END")), None)
    if header_end is None:
        raise ValueError(f"Missing HEADER_END marker in {path}")
    table = pd.read_csv(
        io.StringIO("\n".join(lines[header_end + 1:])),
        sep=r"\s+", comment="#", header=None,
        names=["frequency_mhz", "sensitivity_db", "sensitivity_v_per_pa",
               "sensitivity_v2_cm2_per_w"],
    )
    table = table.apply(pd.to_numeric, errors="raise").sort_values("frequency_mhz").reset_index(drop=True)
    if (table.empty or not np.isfinite(table.to_numpy()).all()
            or table["frequency_mhz"].duplicated().any()
            or (table["sensitivity_v_per_pa"] <= 0).any()):
        raise ValueError(f"Invalid calibration table in {path}")
    return table


def sensitivity_at_frequency(table, frequency_mhz):
    frequency_mhz = float(frequency_mhz)
    frequencies = table["frequency_mhz"].to_numpy()
    sensitivities = table["sensitivity_v_per_pa"].to_numpy()
    if not np.isfinite(frequency_mhz) or not frequencies[0] <= frequency_mhz <= frequencies[-1]:
        raise ValueError(f"Frequency must be within {frequencies[0]}–{frequencies[-1]} MHz")
    upper = int(np.searchsorted(frequencies, frequency_mhz))
    if frequencies[upper] == frequency_mhz:
        return float(sensitivities[upper]), float(frequencies[upper]), float(frequencies[upper])
    lower = upper - 1
    fraction = (frequency_mhz - frequencies[lower]) / (frequencies[upper] - frequencies[lower])
    sensitivity = sensitivities[lower] + fraction * (sensitivities[upper] - sensitivities[lower])
    return float(sensitivity), float(frequencies[lower]), float(frequencies[upper])


calibration_tables = {}
calibration_rows = []
for folder in measurement_dirs:
    if folder.name not in measurement_frequencies_mhz:
        raise ValueError(f"Specify a frequency in MHz for {folder.name}")
    candidates = [folder / (calibration_basename + ".txt"), folder / calibration_basename]
    calibration_path = next((p for p in candidates if p.is_file()), None)
    if calibration_path is None:
        raise FileNotFoundError(f"Calibration file missing in {folder}")
    table = read_calibration(calibration_path)
    calibration_tables[folder.name] = table
    frequency = measurement_frequencies_mhz[folder.name]
    sensitivity, lower, upper = sensitivity_at_frequency(table, frequency)
    calibration_rows.append({
        "measurement": folder.name, "frequency_mhz": float(frequency),
        "sensitivity_v_per_pa": sensitivity,
        "lower_frequency_mhz": lower, "upper_frequency_mhz": upper,
        "interpolated": lower != upper,
        "calibration_file": str(calibration_path.relative_to(measurement_root)),
    })

calibration_df = pd.DataFrame(calibration_rows)
sensitivity_by_measurement = calibration_df.set_index("measurement")["sensitivity_v_per_pa"]
for frame, columns in [
    (average_df, {"Average_Detrended_Amplitude_V": "Average_Detrended_Amplitude_Pa",
                  "A_Mode_Envelope_V": "A_Mode_Envelope_Pa"}),
    (pulses_df, {"Amplitude_V": "Amplitude_Pa"}),
]:
    frame["sensitivity_v_per_pa"] = frame["measurement"].map(sensitivity_by_measurement)
    for voltage_column, pressure_column in columns.items():
        frame[pressure_column] = frame[voltage_column] / frame["sensitivity_v_per_pa"]

display(calibration_df)
display(average_df.head(), pulses_df.head())

,measurement,frequency_mhz,sensitivity_v_per_pa,lower_frequency_mhz,upper_frequency_mhz,interpolated,calibration_file
0,A-mode_2MHz_focus_10Vpp_20cycles,2.00,0.000003,2.00,2.0,False,A-mode_2MHz_focus_10Vpp_20cycles/HGL0400-2875_...
1,A-mode_7p37MHz_focus_10Vpp_20cycles,7.37,0.000003,7.35,7.4,True,A-mode_7p37MHz_focus_10Vpp_20cycles/HGL0400-28...


,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V,sensitivity_v_per_pa,Average_Detrended_Amplitude_Pa,A_Mode_Envelope_Pa
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-2.020624e-07,-0.002025,0.000051,0.000003,-767.147538,19.217160
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.920624e-07,0.005975,0.000047,0.000003,2263.253779,17.669830
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.820624e-07,0.001975,0.000053,0.000003,748.200335,19.935737
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.720624e-07,-0.006024,0.000049,0.000003,-2282.004696,18.416265
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.620624e-07,0.001976,0.000053,0.000003,748.396621,20.015186


,measurement,source_file,pulse_index,Time_s,Amplitude_V,sensitivity_v_per_pa,Amplitude_Pa
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-2.020624e-07,0.000,0.000003,0.000000
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.920624e-07,0.008,0.000003,3030.303174
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.820624e-07,0.004,0.000003,1515.151587
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.720624e-07,-0.004,0.000003,-1515.151587
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.620624e-07,0.004,0.000003,1515.151587


In [6]:
# Example: select one recording and its first pulse for further processing.
measurement = measurement_dirs[1].name
average = average_df.loc[average_df["measurement"].eq(measurement)].copy()
pulse = pulses_df.loc[
    pulses_df["measurement"].eq(measurement) & pulses_df["pulse_index"].eq(1)
].copy()
display(average.head(), pulse.head())

,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V,sensitivity_v_per_pa,Average_Detrended_Amplitude_Pa,A_Mode_Envelope_Pa
10002,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.003104,0.000173,0.000003,1155.462606,64.374513
10003,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.003104,0.000219,0.000003,1155.637074,81.503803
10004,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.001105,0.000156,0.000003,411.209869,57.965861
10005,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.005105,0.000243,0.000003,1900.587684,90.497316
10006,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,-0.012895,0.000138,0.000003,-4800.652910,51.214335


,measurement,source_file,pulse_index,Time_s,Amplitude_V,sensitivity_v_per_pa,Amplitude_Pa
10002,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.006,0.000003,2233.805020
10003,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.006,0.000003,2233.805020
10004,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.004,0.000003,1489.203347
10005,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.008,0.000003,2978.406694
10006,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,-0.010,0.000003,-3723.008367


In [7]:
# Overlay the selected pulse and saved average in Pa.
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=pulse["Time_s"], y=pulse["Amplitude_Pa"],
                mode="lines", name="Pulse 1", opacity=0.7)
fig.add_scatter(x=average["Time_s"], y=average["Average_Detrended_Amplitude_Pa"],
                mode="lines", name="Average (detrended)")
fig.add_scatter(x=average["Time_s"], y=average["A_Mode_Envelope_Pa"],
                mode="lines", name="Envelope")
fig.update_layout(title=measurement, xaxis_title="Time (s)",
                  yaxis_title="Pressure (Pa)", template="plotly_white",
                  hovermode="x unified")
fig.show()

In [8]:
import numpy as np
import scipy.signal as ss
from scipy.fft import rfft, rfftfreq
import plotly.graph_objects as go
from plotly.subplots import make_subplots

hydrophone_bandwidth = 20 #MHz

time_s = pulse["Time_s"].to_numpy(dtype=float)
pressure_pa = pulse["Amplitude_Pa"].to_numpy(dtype=float)

dt = np.median(np.diff(time_s))
if dt <= 0 or not np.allclose(np.diff(time_s), dt, rtol=1e-5, atol=0):
    raise ValueError("Time samples must be uniformly spaced.")

fs = 1 / dt
pressure_pa = ss.detrend(pressure_pa, type="constant")

# Full-pulse FFT: Hann window with coherent-gain amplitude correction.
window = ss.windows.hann(len(pressure_pa), sym=False)
frequency_hz = rfftfreq(len(pressure_pa), d=dt)
amplitude_pa = np.abs(rfft(pressure_pa * window)) / window.sum()

# Convert to a one-sided peak-amplitude spectrum.
amplitude_pa[1:] *= 2
if len(pressure_pa) % 2 == 0:
    amplitude_pa[-1] /= 2  # Nyquist bin is not doubled.

# Time-resolved spectrum.
nperseg = min(256, len(pressure_pa))
f, t, magnitude = ss.spectrogram(
    pressure_pa,
    fs=fs,
    window="hann",
    nperseg=nperseg,
    noverlap=nperseg // 2,
    nfft=512,
    detrend=False,
    scaling="spectrum",
    mode="magnitude",
)

# Spectrogram level relative to its maximum.
reference = magnitude.max()
level_db = 20 * np.log10(
    np.maximum(magnitude / reference, 1e-6)
) if reference > 0 else np.full_like(magnitude, -120.0)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Full-pulse FFT", "Spectrogram"),
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Scatter(
        x=frequency_hz / 1e6,
        y=amplitude_pa / 1000,
        mode="lines",
        name="FFT",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Heatmap(
        x=(t + time_s[0]) * 1e6,
        y=f / 1e6,
        z=level_db,
        colorscale="Viridis",
        zmin=-60, zmax=0,
        colorbar=dict(title="dB re. max", len=0.4, y=0.2),
    ),
    row=2, col=1,
)

fig.update_xaxes(title_text="Frequency (MHz)", row=1, col=1)
fig.update_yaxes(title_text="Peak amplitude (kPa)", row=1, col=1)
fig.update_xaxes(title_text="Time (µs)", row=2, col=1)
fig.update_yaxes(title_text="Frequency (MHz)", row=2, col=1)

fig.update_layout(
    title=f"{measurement} — Pulse spectrum",
    template="plotly_white",
    width=1000,
    height=750,
    showlegend=False,
)

# Display only frequencies within the hydrophone's 20 MHz bandwidth.
fig.update_xaxes(range=[0, hydrophone_bandwidth], row=1, col=1)  # FFT frequency axis
fig.update_yaxes(range=[0, hydrophone_bandwidth], row=2, col=1)  # Spectrogram frequency axis

fig.show()

## A-mode aperture correction and cumulative-energy intensity

A single A-mode trace cannot identify spatial averaging from its 400 µm electrode aperture alone. Supply four measured off-axis/on-axis pressure-like ratios or use the **matching pressure-field scan** below. Scan-derived factors are provisional: the grid is coarse, the effective aperture is assumed, and the A-mode recording must correspond to the same beam geometry and spatial peak. The notebook cannot establish that correspondence automatically. A field-wide or nonlinear waveform correction is not inferred.

Under the user-assumed linear, separable beam, use **one scalar pressure factor** from the spatial profile of `sqrt(PII)`: `K = (3 − beta)/2`, with beta averaged at ± one effective radius in two perpendicular directions. Apply this factor to the selected pulse, average and envelope. This produces a model-corrected waveform estimate; using different compressional/rarefactional peak factors piecewise would distort the waveform. Intensity/PII then scale by K² and cumulative-energy PD is unchanged.

The quadratic method is associated with IEC 62127-1:2007 + Amendment 1:2013 Annex E; this is not a 2022-edition compliance claim. [Derivation](https://pmc.ncbi.nlm.nih.gov/articles/PMC6935508/), [Onda electrode specifications](https://www.ondacorp.com/capsule-hydrophone/).

Intensity uses `I(t) = p²/(rho*c)`; full gated PII and central-98% PII are separate; `PD = 1.25*(t90 − t10)`; `I_PA = selected_PII/PD`; repeating-train `I_TA = selected_PII*PRF`. These are **point quantities**, not I_SPPA/I_SPTA unless this location is established as the spatial peak. Finite-exposure averaging is also reported.

The gate and pre-pulse noise interval must be checked at the selected frequency. Default gates below suit the observed records, not arbitrary recordings. Remove baseline mean and optionally subtract expected noise energy. Signed estimates are retained; unresolved PD/I_PA are NaN. Monotonic cumulative-energy projection for noise-adjusted quantiles is an explicit statistical estimator, not an IEC-prescribed procedure. Do not clip each squared-pressure-minus-variance sample to zero.

The saved envelope is processed by acquisition software and need not equal the analytic envelope of the unfiltered waveform. Both are labeled separately. Noise-energy subtraction does not reconstruct a noise-free instantaneous pressure waveform.

In [9]:
# USER SETTINGS: selected measurement is set in the preceding selection cell.
measurement_frequency_mhz = float(measurement_frequencies_mhz[measurement])
no_cycles = 20
no_pulses = 1
density_kg_m3 = 998.0
sound_speed_m_s = 1482.0
# The A-mode header records PRF; override this value if acquisition information differs.
selected_metadata = metadata_df.loc[
    metadata_df["measurement"].eq(measurement) & metadata_df["reading_type"].eq("average")
].iloc[0]
prf_hz = float(selected_metadata["prf_hz"])
averaging_duration_s = None  # None: no_pulses / prf_hz.

pulse_gates_by_measurement = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": (42e-6, 52e-6),
    "A-mode_2MHz_focus_10Vpp_20cycles": (40e-6, 60e-6),
}
noise_windows_by_measurement = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": (40.05e-6, 42e-6),
    "A-mode_2MHz_focus_10Vpp_20cycles": (35e-6, 39e-6),
}
pulse_gate_s = pulse_gates_by_measurement[measurement]
noise_window_s = noise_windows_by_measurement[measurement]
subtract_noise_energy = True
minimum_signal_to_noise_energy_ratio = 3.0
pii_convention = "full_pulse"  # Or "central_98_percent".

electrode_aperture_diameter_um = 400.0
effective_aperture_diameter_um = None  # Override with effective diameter at this frequency.
# None: derive an estimate from the matching scan. Otherwise supply four ratios in (0,1].
measured_aperture_ratios = None  # [axis1_minus, axis1_plus, axis2_minus, axis2_plus]
pressure_field_folders = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": "pressure map_7p37MHz_10Vpp_focal plane",
    "A-mode_2MHz_focus_10Vpp_20cycles": "pressure map_2MHz_10Vpp_ focal plane",
}
# This assumption must be physically checked before using the scan-derived factor.
assume_same_beam_and_spatial_peak = True
allow_interpolated_aperture_estimate = True


In [10]:
import scipy.signal as ss
from scipy.optimize import isotonic_regression

def pulse_energy_metrics(pressure, times, impedance, gate=None,
                         noise_variance_pa2=0.0, subtract_noise=False, minimum_energy_snr=3.0):
    """Pulse energy with expected noise subtraction and optional quantile reliability mask.

    Signed cumulative energy is projected monotonically for quantiles only.
    No sample-by-sample clipping of p²-noise variance is performed.
    """
    from scipy.optimize import isotonic_regression
    pressure, times = np.asarray(pressure, float), np.asarray(times, float)
    if pressure.shape != times.shape or len(times) < 2 or not np.isfinite(pressure).all():
        raise ValueError("Expected a finite 1-D waveform and matching times")
    if not np.isfinite(times).all() or (np.diff(times) <= 0).any() or not np.isfinite(impedance) or impedance <= 0:
        raise ValueError("Invalid time axis or acoustic impedance")
    if not np.isfinite(noise_variance_pa2) or noise_variance_pa2 < 0:
        raise ValueError("Noise variance must be finite and nonnegative")
    start, end = (times[0], times[-1]) if gate is None else tuple(gate)
    if not np.isfinite([start, end]).all() or not times[0] <= start < end <= times[-1]:
        raise ValueError("Pulse gate must lie inside the acquired time interval")
    intensity = pressure ** 2 / impedance
    gated_time = np.r_[start, times[(times > start) & (times < end)], end]
    gated_intensity = np.interp(gated_time, times, intensity)
    raw_areas = 0.5 * (gated_intensity[:-1] + gated_intensity[1:]) * np.diff(gated_time)
    raw_cumulative = np.r_[0., np.cumsum(raw_areas)]
    noise_intensity = noise_variance_pa2 / impedance
    noise_energy = noise_intensity * (end - start)
    signed_net = float(raw_cumulative[-1] - noise_energy)
    energy_snr = signed_net / noise_energy if noise_energy > 0 else np.inf if raw_cumulative[-1] > 0 else 0.
    signed_cumulative = raw_cumulative - (noise_intensity * (gated_time - start) if subtract_noise else 0.)
    signed_total = float(signed_cumulative[-1])
    # Nonnegative per-point display estimate. Retain signed values for unbiased spatial sums.
    total = max(signed_total, 0.)
    reliable = bool(total > 0 and energy_snr >= minimum_energy_snr)
    result = dict(pii_full=total, pii_98=np.nan,
                  t01=np.nan, t10=np.nan, t90=np.nan, t99=np.nan, pd=np.nan,
                  gate_start=start, gate_end=end,
                  pii_raw=float(raw_cumulative[-1]), pii_signed=signed_total,
                  expected_noise_pii=noise_energy, energy_snr=energy_snr,
                  duration_reliable=float(reliable), cumulative_projection_change=np.nan)
    if not reliable:
        return result
    # Energy after subtraction is signed and need not increase monotonically.
    # Fit a nondecreasing cumulative curve, constrain it to [0,total], and pin its endpoints.
    cumulative = np.clip(isotonic_regression(signed_cumulative).x, 0., total)
    cumulative[0], cumulative[-1] = 0., total
    result["cumulative_projection_change"] = float(np.max(np.abs(cumulative - signed_cumulative)) / total)
    # Invert the projected cumulative curve; fractional sample timing is interpolated.
    targets = np.array([.01, .1, .9, .99]) * total
    segments = np.clip(np.searchsorted(cumulative, targets, side="right") - 1, 0, len(cumulative) - 2)
    fractions = (targets - cumulative[segments]) / (cumulative[segments + 1] - cumulative[segments])
    t01, t10, t90, t99 = gated_time[segments] + fractions * (gated_time[segments + 1] - gated_time[segments])
    result.update(pii_98=.98 * total, t01=t01, t10=t10, t90=t90, t99=t99,
                  pd=1.25 * (t90 - t10))
    return result

def annex_e_peak_estimate(signal_map, axis1, axis2, radius_mm,
                          measured_ratios=None, allow_interpolation=True):
    """Near-axis quadratic correction for a nonnegative pressure-like quantity.

    Returns a spatial-peak estimate only; no correction is applied off-axis.
    """
    values = np.asarray(signal_map, dtype=float)
    axis1, axis2 = np.asarray(axis1, dtype=float), np.asarray(axis2, dtype=float)
    if values.shape != (len(axis1), len(axis2)) or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Expected a finite nonnegative map matching the scan axes")
    if not np.isfinite(radius_mm) or radius_mm <= 0:
        raise ValueError("Radius must be finite and positive")
    if len(axis1) < 2 or len(axis2) < 2 or (np.diff(axis1) <= 0).any() or (np.diff(axis2) <= 0).any():
        raise ValueError("Two increasing spatial axes are required")
    i, j = np.unravel_index(np.argmax(values), values.shape)
    peak = float(values[i, j])
    coarse = bool(max(np.diff(axis1).max(), np.diff(axis2).max()) > radius_mm)
    result = {"axis1_index": int(i), "axis2_index": int(j),
              "axis1_mm": float(axis1[i]), "axis2_mm": float(axis2[j]),
              "measured_peak": peak, "radius_mm": radius_mm,
              "scan_coarser_than_radius": coarse,
              "effective_diameter_assumed": effective_aperture_diameter_um is None}
    if peak == 0:
        return {**result, "beta": np.nan, "pressure_factor": 1.0,
                "corrected_peak": 0.0, "status": "zero signal", "source": "none"}
    if measured_ratios is None and np.all(values == peak):
        return {**result, "beta": 1.0, "pressure_factor": 1.0,
                "corrected_peak": peak, "status": "uniform sampled field; no estimated correction",
                "source": "uniform sampled profile"}
    if measured_ratios is not None:
        ratios = np.asarray(measured_ratios, dtype=float)
        if ratios.shape != (4,) or not np.isfinite(ratios).all() or (ratios <= 0).any() or (ratios > 1).any():
            raise ValueError("Supply four ratios in (0, 1], measured around the quantity's peak")
        source = "user-supplied measurements"
    else:
        if coarse and not allow_interpolation:
            raise ValueError("Scan spacing exceeds hydrophone radius; acquire a finer scan or supply measured ratios")
        x, y = axis1[i], axis2[j]
        if x - radius_mm < axis1[0] or x + radius_mm > axis1[-1] or y - radius_mm < axis2[0] or y + radius_mm > axis2[-1]:
            return {**result, "beta": np.nan, "pressure_factor": np.nan,
                    "corrected_peak": np.nan, "status": "insufficient spatial coverage", "source": "none"}
        ratios = np.r_[np.interp([x - radius_mm, x + radius_mm], axis1, values[:, j]),
                       np.interp([y - radius_mm, y + radius_mm], axis2, values[i, :])] / peak
        source = "interpolated scan profile"
    beta = float(ratios.mean())
    factor = (3.0 - beta) / 2.0
    drops_db = -20 * np.log10(np.maximum(ratios, np.finfo(float).tiny))
    status = "quadratic-model estimate"
    if measured_ratios is None and coarse:
        status = "provisional: coarse scan interpolation"
    if (ratios <= 0).any() or (ratios > 1).any():
        status = "invalid off-axis ratios"
        factor = np.nan
    return {**result, "beta": beta, "pressure_factor": factor,
            "corrected_peak": peak * factor, "status": status, "source": source,
            "beta_axis1_minus": ratios[0], "beta_axis1_plus": ratios[1],
            "beta_axis2_minus": ratios[2], "beta_axis2_plus": ratios[3],
            "max_off_axis_drop_db": float(drops_db.max()),
            "off_axis_drop_exceeds_1db": bool((drops_db > 1).any())}


def process_point_waveform(frame, pressure_column, pressure_factor=1.0):
    times = frame["Time_s"].to_numpy(dtype=float)
    values = frame[pressure_column].to_numpy(dtype=float)
    if len(times) < 2 or not np.allclose(np.diff(times), np.median(np.diff(times)), rtol=1e-5, atol=0):
        raise ValueError("Expected uniformly spaced samples")
    if not times[0] <= noise_window_s[0] < noise_window_s[1] <= times[-1]:
        raise ValueError("Baseline window must lie within this waveform")
    if max(noise_window_s[0], pulse_gate_s[0]) < min(noise_window_s[1], pulse_gate_s[1]):
        raise ValueError("Pulse and baseline windows must not overlap")
    noise_mask = (times >= noise_window_s[0]) & (times <= noise_window_s[1])
    if noise_mask.sum() < 20:
        raise ValueError("Insufficient baseline samples")
    baseline = float(np.mean(values[noise_mask]))
    variance = float(np.var(values[noise_mask], ddof=1))
    centred = values - baseline
    result = pulse_energy_metrics(centred * pressure_factor, times, acoustic_impedance_pa_s_m,
                                 pulse_gate_s, variance * pressure_factor**2,
                                 subtract_noise_energy, minimum_signal_to_noise_energy_ratio)
    pii = result["pii_full"] if pii_convention == "full_pulse" else result["pii_98"]
    duration = result["pd"]
    result.update(
        selected_pii_j_m2=pii, selected_pii_mj_cm2=pii / 10,
        pulse_average_intensity_w_cm2=pii / duration / 1e4 if np.isfinite(duration) and duration > 0 else np.nan,
        temporal_average_intensity_w_cm2=pii * prf_hz / 1e4,
        finite_exposure_intensity_w_cm2=no_pulses * pii / temporal_averaging_duration_s / 1e4,
        peak_compressional_kpa=max(float(np.max(centred)), 0) * pressure_factor / 1000,
        peak_rarefactional_magnitude_kpa=max(float(-np.min(centred)), 0) * pressure_factor / 1000,
        baseline_pressure_pa=baseline, noise_variance_pa2=variance * pressure_factor**2,
    )
    return result, centred * pressure_factor


In [11]:
if not all(np.isfinite(value) and value > 0 for value in
           [measurement_frequency_mhz, no_cycles, no_pulses, prf_hz, density_kg_m3,
            sound_speed_m_s, minimum_signal_to_noise_energy_ratio]):
    raise ValueError("Frequency, pulse parameters, water properties and SNR threshold must be positive")
if int(no_pulses) != no_pulses:
    raise ValueError("no_pulses must be an integer")
if pii_convention not in {"full_pulse", "central_98_percent"}:
    raise ValueError("Unknown PII convention")
acoustic_impedance_pa_s_m = density_kg_m3 * sound_speed_m_s
nominal_pulse_duration_s = no_cycles / (measurement_frequency_mhz * 1e6)
temporal_averaging_duration_s = (no_pulses / prf_hz if averaging_duration_s is None
                                else float(averaging_duration_s))
if not np.isfinite(temporal_averaging_duration_s) or temporal_averaging_duration_s <= 0:
    raise ValueError("Exposure averaging duration must be positive")
aperture_diameter_um = (electrode_aperture_diameter_um if effective_aperture_diameter_um is None
                        else float(effective_aperture_diameter_um))
if not np.isfinite(aperture_diameter_um) or aperture_diameter_um <= 0:
    raise ValueError("Effective aperture diameter must be positive")
effective_radius_mm = aperture_diameter_um / 2000

if measured_aperture_ratios is not None:
    ratios = np.asarray(measured_aperture_ratios, dtype=float)
    if ratios.shape != (4,) or not np.isfinite(ratios).all() or (ratios <= 0).any() or (ratios > 1).any():
        raise ValueError("Supply four pressure-like ratios in (0,1]")
    beta = float(ratios.mean())
    aperture_pressure_factor = (3 - beta) / 2
    aperture_diagnostics = {"beta": beta, "pressure_factor": aperture_pressure_factor,
                            "source": "user-supplied ratios", "status": "quadratic-model estimate"}
else:
    if not assume_same_beam_and_spatial_peak:
        raise ValueError("Supply measured ratios unless the A-mode trace shares the scan beam and spatial peak")
    scan_root = measurement_root / pressure_field_folders[measurement]
    scan_manifest_df = pd.read_csv(scan_root / "point_manifest.csv")
    if set(scan_manifest_df["col_axis_role"]) != {"axis_1"} or set(scan_manifest_df["row_axis_role"]) != {"axis_2"}:
        raise ValueError("Expected column axis 1 and row axis 2 in matching scan")
    scan_axis1_mm = np.sort(scan_manifest_df["scan_mm"].unique())
    scan_axis2_mm = np.sort(scan_manifest_df["cross_mm"].unique())
    if len(scan_manifest_df) != len(scan_axis1_mm) * len(scan_axis2_mm) or scan_manifest_df.duplicated(["scan_mm", "cross_mm"]).any():
        raise ValueError("Matching scan must be a complete grid without duplicate coordinates")
    calibration_candidates = [scan_root / (calibration_basename + ".txt"), scan_root / calibration_basename]
    scan_calibration_path = next((path for path in calibration_candidates if path.is_file()), None)
    if scan_calibration_path is None:
        raise FileNotFoundError("Matching scan calibration is missing")
    scan_sensitivity_v_per_pa = sensitivity_at_frequency(read_calibration(scan_calibration_path), measurement_frequency_mhz)[0]
    scan_waveforms = []
    for _, point in scan_manifest_df.iterrows():
        frame = pd.read_csv(scan_root / point["measurement_csv"], comment="#")
        frame = frame.rename(columns={"Time (s)": "Time_s", "Amplitude (V)": "Amplitude_V"})
        frame["Amplitude_Pa"] = frame["Amplitude_V"] / scan_sensitivity_v_per_pa
        scan_waveforms.append(frame)
    reference_times = scan_waveforms[0]["Time_s"].to_numpy()
    overlap_start = max(frame["Time_s"].iloc[0] for frame in scan_waveforms)
    overlap_end = min(frame["Time_s"].iloc[-1] for frame in scan_waveforms)
    scan_time_s = reference_times[(reference_times >= overlap_start) & (reference_times <= overlap_end)]
    scan_sqrt_ppsi_map = np.empty((len(scan_axis1_mm), len(scan_axis2_mm)))
    for (_, point), original in zip(scan_manifest_df.iterrows(), scan_waveforms):
        aligned = pd.DataFrame({"Time_s": scan_time_s,
                                "Amplitude_Pa": np.interp(scan_time_s, original["Time_s"], original["Amplitude_Pa"])})
        result, _ = process_point_waveform(aligned, "Amplitude_Pa")
        i = int(np.searchsorted(scan_axis1_mm, point["scan_mm"]))
        j = int(np.searchsorted(scan_axis2_mm, point["cross_mm"]))
        scan_sqrt_ppsi_map[i, j] = np.sqrt(max(result["selected_pii_j_m2"], 0) * acoustic_impedance_pa_s_m)
    if not np.isfinite(scan_sqrt_ppsi_map).all():
        raise ValueError("Selected PII convention is unresolved at some scan points; use full-pulse PII")
    aperture_diagnostics = annex_e_peak_estimate(
        scan_sqrt_ppsi_map, scan_axis1_mm, scan_axis2_mm, effective_radius_mm,
        allow_interpolation=allow_interpolated_aperture_estimate)
    aperture_diagnostics["scan_folder"] = str(scan_root.relative_to(project_root))
    aperture_diagnostics["assumes_same_beam_and_spatial_peak"] = True
    aperture_pressure_factor = aperture_diagnostics["pressure_factor"]
if not np.isfinite(aperture_pressure_factor) or aperture_pressure_factor < 1:
    raise ValueError("A valid spatial-peak correction factor could not be established")
aperture_intensity_factor = aperture_pressure_factor**2
aperture_correction_df = pd.DataFrame([{**aperture_diagnostics,
                                       "effective_radius_mm": effective_radius_mm,
                                       "intensity_factor": aperture_intensity_factor,
                                       "frequency_mhz": measurement_frequency_mhz}])
display(aperture_correction_df)


,axis1_index,axis2_index,axis1_mm,axis2_mm,measured_peak,radius_mm,scan_coarser_than_radius,effective_diameter_assumed,beta,pressure_factor,...,beta_axis1_plus,beta_axis2_minus,beta_axis2_plus,max_off_axis_drop_db,off_axis_drop_exceeds_1db,scan_folder,assumes_same_beam_and_spatial_peak,effective_radius_mm,intensity_factor,frequency_mhz
0,6,6,2.181818,2.181818,240.393091,0.2,True,True,0.613716,1.193142,...,0.629312,0.575222,0.600741,4.803293,True,data/Measurements - Onda hydrophone data - 202...,True,0.2,1.423588,7.37


In [12]:
# Preserve original DataFrames and store model-corrected, baseline-centred estimates separately.
pulse_corrected = pulse.copy()
average_corrected = average.copy()
intensity_rows = []
for label, frame, column in [
    ("Selected pulse", pulse, "Amplitude_Pa"),
    ("Saved average", average, "Average_Detrended_Amplitude_Pa"),
]:
    raw, centred = process_point_waveform(frame, column)
    corrected, corrected_waveform = process_point_waveform(frame, column, aperture_pressure_factor)
    for kind, result in [("Measured (baseline-centred)", raw), ("Aperture-corrected estimate", corrected)]:
        intensity_rows.append({"trace": label, "processing": kind, **result,
                               "prf_hz": prf_hz, "frequency_mhz": measurement_frequency_mhz,
                               "nominal_pulse_duration_s": nominal_pulse_duration_s,
                               "pii_convention": pii_convention})
    if label == "Selected pulse":
        pulse_corrected["Baseline_Centred_Pressure_Pa"] = centred
        pulse_corrected["Corrected_Amplitude_Pa"] = corrected_waveform
        pulse_corrected["Corrected_Analytic_Envelope_Pa"] = np.abs(ss.hilbert(corrected_waveform))
        pulse_corrected["Corrected_Instantaneous_Intensity_W_m2"] = corrected_waveform**2 / acoustic_impedance_pa_s_m
        pulse_energy_result = corrected
    else:
        average_corrected["Baseline_Centred_Pressure_Pa"] = centred
        average_corrected["Corrected_Average_Pressure_Pa"] = corrected_waveform
        average_corrected["Corrected_Saved_Envelope_Pa"] = frame["A_Mode_Envelope_Pa"].to_numpy() * aperture_pressure_factor
        average_corrected["Corrected_Analytic_Envelope_Pa"] = np.abs(ss.hilbert(corrected_waveform))
        average_corrected["Corrected_Instantaneous_Intensity_W_m2"] = corrected_waveform**2 / acoustic_impedance_pa_s_m

intensity_report_df = pd.DataFrame(intensity_rows)
valid_durations = intensity_report_df["pd"].dropna()
if not valid_durations.empty:
    minimum_exposure = (no_pulses - 1) / prf_hz + valid_durations.max()
    if valid_durations.max() * prf_hz > 1 or temporal_averaging_duration_s < minimum_exposure:
        raise ValueError("PRF or exposure averaging duration is inconsistent with measured PD")
intensity_report_df["PD_us"] = intensity_report_df["pd"] * 1e6
report_columns = ["trace", "processing", "peak_compressional_kpa", "peak_rarefactional_magnitude_kpa",
                  "PD_us", "selected_pii_mj_cm2", "pulse_average_intensity_w_cm2",
                  "temporal_average_intensity_w_cm2", "finite_exposure_intensity_w_cm2",
                  "energy_snr", "duration_reliable"]
display(intensity_report_df[report_columns])

# Individual-pulse metrics: averaging p²-based metrics retains pulse-to-pulse variation.
per_pulse_rows = []
for pulse_index, frame in pulses_df.loc[pulses_df["measurement"].eq(measurement)].groupby("pulse_index"):
    result, _ = process_point_waveform(frame, "Amplitude_Pa", aperture_pressure_factor)
    per_pulse_rows.append({"pulse_index": pulse_index, **result})
per_pulse_intensity_df = pd.DataFrame(per_pulse_rows)
display(per_pulse_intensity_df[["pulse_index", "pd", "selected_pii_mj_cm2",
                               "pulse_average_intensity_w_cm2", "temporal_average_intensity_w_cm2"]])


,trace,processing,peak_compressional_kpa,peak_rarefactional_magnitude_kpa,PD_us,selected_pii_mj_cm2,pulse_average_intensity_w_cm2,temporal_average_intensity_w_cm2,finite_exposure_intensity_w_cm2,energy_snr,duration_reliable
0,Selected pulse,Measured (baseline-centred),191.416089,195.032180,2.659258,0.002587,0.972828,0.000259,0.000259,1342.876145,1.0
1,Selected pulse,Aperture-corrected estimate,228.386600,232.701111,2.659258,0.003683,1.384906,0.000368,0.000368,1342.876145,1.0
2,Saved average,Measured (baseline-centred),191.496169,194.948610,2.659168,0.002587,0.972820,0.000259,0.000259,1343.854013,1.0
3,Saved average,Aperture-corrected estimate,228.482148,232.601400,2.659168,0.003683,1.384895,0.000368,0.000368,1343.854013,1.0


,pulse_index,pd,selected_pii_mj_cm2,pulse_average_intensity_w_cm2,temporal_average_intensity_w_cm2
0,1,0.000003,0.003683,1.384906,0.000368


In [13]:
from plotly.subplots import make_subplots

fig_corrected_a_mode = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.09,
    subplot_titles=("Aperture-corrected pressure estimate", "Corrected envelopes",
                    "Selected pulse cumulative energy"))
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Baseline_Centred_Pressure_Pa"] / 1000,
    name="Measured pulse", opacity=.45, row=1, col=1)
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Corrected_Amplitude_Pa"] / 1000,
    name="Corrected pulse estimate", row=1, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Average_Pressure_Pa"] / 1000,
    name="Corrected average estimate", row=1, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Saved_Envelope_Pa"] / 1000,
    name="Corrected saved envelope", row=2, col=1)
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Corrected_Analytic_Envelope_Pa"] / 1000,
    name="Corrected pulse analytic envelope", row=2, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Analytic_Envelope_Pa"] / 1000,
    name="Corrected average analytic envelope", row=2, col=1)

times = pulse_corrected["Time_s"].to_numpy()
start, end = pulse_gate_s
gated_times = np.r_[start, times[(times > start) & (times < end)], end]
gated_intensity = np.interp(gated_times, times, pulse_corrected["Corrected_Instantaneous_Intensity_W_m2"])
raw_cumulative = np.r_[0., np.cumsum(.5 * (gated_intensity[:-1] + gated_intensity[1:]) * np.diff(gated_times))]
noise_intensity = pulse_energy_result["noise_variance_pa2"] / acoustic_impedance_pa_s_m
signed_cumulative = raw_cumulative - (noise_intensity * (gated_times-start) if subtract_noise_energy else 0)
total = max(float(signed_cumulative[-1]), 0.)
projected = np.clip(isotonic_regression(signed_cumulative).x, 0., total)
projected[0], projected[-1] = 0., total
energy_fraction = projected / total if total > 0 else np.full_like(projected, np.nan)
fig_corrected_a_mode.add_scatter(x=gated_times * 1e6, y=energy_fraction,
                               name="Noise-adjusted cumulative energy", row=3, col=1)
for key in ["t01", "t10", "t90", "t99"]:
    value = pulse_energy_result[key]
    if np.isfinite(value):
        fig_corrected_a_mode.add_vline(x=value * 1e6, line_dash="dot", annotation_text=key, row=3, col=1)
for row in [1, 2]:
    fig_corrected_a_mode.add_vrect(x0=start*1e6, x1=end*1e6, fillcolor="lightgray",
                                  opacity=.12, line_width=0, row=row, col=1)
    fig_corrected_a_mode.update_yaxes(title_text="Pressure (kPa)", row=row, col=1)
fig_corrected_a_mode.update_yaxes(title_text="Energy fraction", row=3, col=1)
fig_corrected_a_mode.update_xaxes(title_text="Time (µs)", range=[start*1e6-1, end*1e6+1], row=3, col=1)
fig_corrected_a_mode.update_layout(
    title=f"{measurement} — provisional aperture factor {aperture_pressure_factor:.4f}",
    template="plotly_white", width=1100, height=950, hovermode="x unified")
fig_corrected_a_mode.show()
